# Semana 04 — Manipulação de Arquivos e Modularização

**Curso:** Análise de Dados com Python — SENAI (Turma T5)
**UC (MSEP):** Manipulação de Dados com Python e SQL (150h) — Bloco 2, Semanas 03 e 04 (última semana do bloco)

Esta semana fecha o Bloco 2: você veste o boné de analista júnior de **People Analytics** do **Grupo Alfa** (empresa fictícia, 30 colaboradores, 4 setores) e trabalha com 3 arquivos reais que o RH te passou — além de aprender a organizar seu código em funções e módulos reutilizáveis.

> Em cada tópico abaixo: **exemplos resolvidos** + **atividades práticas** para você fazer sozinho(a).

---
### 🟢 Abertura — Semana 04: Manipulação de Arquivos e Modularização

Você já versiona código. **Hoje você veste o boné de analista júnior de People Analytics** — em português, algo como "análise de pessoas": a área de uma empresa que usa dados sobre os próprios funcionários (horário de entrada, atrasos, cadastro, e-mails) para tomar decisões, em vez de decidir só no "achismo". É a área que, por exemplo, descobre se um setor inteiro está sempre atrasado, ou se um cadastro está desatualizado.

Você vai ler dados de arquivos reais do Grupo Alfa e organizar seu código em funções reutilizáveis.

**O que você vai aprender hoje:**
- Ler e escrever arquivos CSV, JSON e Excel com Python
- Trabalhar com datas e expressões regulares em cima de dados reais de ponto e e-mail
- Criar funções com parâmetros, valores padrão e `return`, e usar lambda e módulos

### 📌 Antes de começar: os 3 arquivos desta semana

O RH do Grupo Alfa te passou 3 arquivos (nesta mesma pasta):

| Arquivo | Formato | Sistema de origem |
|---|---|---|
| `cadastro_funcionario.csv` | CSV | Sistema de folha de pagamento |
| `emails_corporativos.json` | JSON | Sistema de e-mail corporativo |
| `recursos_humanos.xlsx` | Excel (2 abas) | Sistema de ponto |

**Rodando no Google Colab?** Diferente do VS Code local, o Colab não enxerga os arquivos desta pasta automaticamente — rode a célula abaixo pra enviá-los.

In [1]:
# Se estiver rodando no Google Colab, esta célula abre a janela de upload.
# Selecione os 3 arquivos desta pasta: cadastro_funcionario.csv, emails_corporativos.json, recursos_humanos.xlsx
try:
    from google.colab import files
    enviados = files.upload()
except ImportError:
    print("Rodando localmente (VS Code) — os arquivos já estão na mesma pasta do notebook.")

Rodando localmente (VS Code) — os arquivos já estão na mesma pasta do notebook.


---
## 1. Arquivos: CSV, JSON e Excel

Até agora, todo dado que você usou existia só enquanto o programa rodava, ou foi digitado à mão no próprio código. A partir de hoje isso muda: cada um dos 3 arquivos do Grupo Alfa vem de um sistema diferente, por isso está num formato diferente — CSV quando o dado é tabular e simples, JSON quando tem estrutura mais livre, Excel quando vem de planilhas com múltiplas abas.

💡 **O que é uma "biblioteca"?** É um conjunto de ferramentas prontas que alguém já programou pra você não precisar reinventar a roda. `csv`, `json` e `re` já vêm **dentro do Python** — basta dar `import`, sem instalar nada. Já `openpyxl` (que você vai usar daqui a pouco pra ler o Excel) é uma biblioteca **de terceiros**: não vem com o Python, então precisa ser **instalada** antes do primeiro uso — você vai ver como fazer isso logo abaixo, no Exemplo 3. (Mais pra frente, na Seção 4, você vai aprender a criar a sua própria biblioteca — lá ela vai se chamar **módulo**: um módulo nada mais é do que uma biblioteca que você mesmo escreveu.)

### 📖 Antes do primeiro código: o que significa "abrir" um arquivo?

Assim como você abre um arquivo do Windows dando duplo clique nele, o Python também precisa "abrir" um arquivo antes de conseguir ler o que está escrito dentro. E, do mesmo jeito que é boa prática fechar um programa quando termina de usar, o Python também precisa **fechar** o arquivo depois de ler — senão ele fica "reservado", e nenhum outro programa consegue mexer nele enquanto isso. Em Python, isso é feito assim:

```python
with open("cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    # aqui dentro, o arquivo está aberto e disponível
    ...
# aqui fora, o Python já fechou o arquivo sozinho
```

Repare em 3 coisas:
- `open("cadastro_funcionario.csv", encoding="utf-8")` é a parte que efetivamente abre o arquivo. Você já vai entender o `encoding="utf-8"` em instantes.
- `as arquivo` dá um apelido pro arquivo aberto — você poderia chamar de qualquer nome (`f`, `meu_arquivo`...), mas `arquivo` deixa claro o que é.
- A palavra `with` é quem garante que o Python vai **fechar o arquivo automaticamente** assim que o bloco indentado (a parte recuada, embaixo dos dois-pontos) terminar — mesmo se algo der errado no meio do caminho. Sempre que for ler ou escrever um arquivo em Python, você vai começar com `with open(...) as ...:`. Decore essa estrutura: ela vai aparecer o resto do curso inteiro.

### 🔹 Exemplo 1 — Lendo o cadastro de funcionários (CSV)

In [4]:
import csv

with open("cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    funcionarios = list(leitor)

print(f"Total de funcionários: {len(funcionarios)}")
print(funcionarios[0])

Total de funcionários: 30
{'id_funcionario': '1', 'funcionario': 'Ana Souza', 'cargo': 'Atendente', 'setor': 'Comercial'}


### 🔹 Exemplo 2 — Lendo os e-mails corporativos (JSON)

In [5]:
import json

with open("emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

print(emails[0])

#for email in emails:
#    print(email)

{'id_funcionario': 1, 'funcionario': 'Ana Souza', 'email': 'ana.souza@grupoalfa.com.br'}


> ⚠️ **Veja como é um erro real do Python.** Tente rodar `open("nao_existe.json", "r")` numa célula nova — você vai ver `FileNotFoundError: [Errno 2] No such file or directory: 'nao_existe.json'`. É o erro mais comum de quem trabalha com arquivos (nome digitado errado, ou arquivo que ainda não existe). Você pode se proteger com `try`/`except` — veja a célula abaixo.

In [6]:
try:
    with open("nao_existe.json", "r", encoding="utf-8") as arquivo:
        dados = json.load(arquivo)
except FileNotFoundError:
    print("Arquivo não encontrado — verifique o nome ou crie o arquivo primeiro.")

Arquivo não encontrado — verifique o nome ou crie o arquivo primeiro.


> ⚠️ **Cuidado com o `encoding`.** Repare que os dois exemplos acima usam `encoding="utf-8"`. Se você esquecer esse parâmetro no Windows, o Python pode abrir o arquivo com outra codificação padrão do sistema — e palavras acentuadas quebram silenciosamente, sem gerar erro nenhum: `"Logística"` pode virar `"LogÃ­stica"` na tela. É um dos bugs mais chatos de encontrar porque **o programa não trava, só mostra o dado errado**.

### 🔹 Exemplo extra — Escrevendo um CSV

Até agora você só **leu** arquivos. Mas o RH também vai pedir pra você **gerar** um novo CSV — por exemplo, uma lista só com os funcionários de um setor. Para escrever, o Python usa `csv.DictWriter`, o "irmão" de escrita do `csv.DictReader` que você já usou:

In [7]:
import csv

with open("cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))

funcionarios_estoque = [f for f in funcionarios if f["setor"] == "Estoque"]
#print(funcionarios_estoque)


with open("funcionarios_estoque.csv", "w", encoding="utf-8", newline="") as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=funcionarios[0].keys())
    escritor.writeheader()
    escritor.writerows(funcionarios_estoque)

print(f"{len(funcionarios_estoque)} funcionários do Estoque salvos em funcionarios_estoque.csv")

9 funcionários do Estoque salvos em funcionarios_estoque.csv


💡 **O que aconteceu em cada linha:**
- `"w"` no `open(...)` significa **escrita** (`write`) — diferente do modo de leitura que você já usa. Cuidado: `"w"` **sobrescreve** o arquivo inteiro se ele já existir.
- `newline=""` é importante ao escrever CSV, principalmente **no Windows (VS Code local)**: sem ele, o módulo `csv` pode inserir uma linha em branco entre cada registro — um bug clássico de Windows. No Colab (que roda em Linux) isso costuma não aparecer, mas escrever sempre com `newline=""` evita o problema nos dois ambientes.
- `csv.DictWriter(arquivo, fieldnames=...)` precisa saber, de antemão, quais são as colunas — por isso passamos `funcionarios[0].keys()`, as mesmas chaves que já vieram do CSV original.
- `.writeheader()` escreve a primeira linha (os nomes das colunas); `.writerows(...)` escreve o restante, um dicionário por linha.

### 🔹 Exemplo 3 — Lendo o banco de horas (Excel, múltiplas abas)

📦 **Antes do código: instalando o `openpyxl`.** Diferente de `csv`, `json`, `re` e `datetime` (que já vêm dentro do Python), `openpyxl` é uma biblioteca **de terceiros** — feita por outras pessoas, não pelo Python — e por isso precisa ser **instalada** antes do primeiro uso. Rode a célula abaixo uma vez: ela funciona tanto no Colab quanto no VS Code local, e se o `openpyxl` já estiver instalado, ela simplesmente não faz nada (não tem problema rodar de novo).

> ⚠️ **Se você pular esta célula**, ao rodar o próximo bloco de código vai aparecer `ModuleNotFoundError: No module named 'openpyxl'`. Se isso acontecer, é só voltar aqui e rodar esta célula antes de tentar de novo.

In [8]:
%pip install -q openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from openpyxl import load_workbook

planilha = load_workbook("recursos_humanos.xlsx")
print("Abas disponíveis:", planilha.sheetnames)



Abas disponíveis: ['cadastro_funcionario', 'banco_horas']


In [10]:
aba_ponto = planilha["banco_horas"]
print(f"Total de registros de ponto: {aba_ponto.max_row - 1}")  # -1 por causa do cabeçalho



Total de registros de ponto: 100


In [11]:
for linha in aba_ponto.iter_rows(min_row=2, max_row=4, values_only=True):
    print(linha)

(1, 'Atendente', 'Comercial', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 30), datetime.datetime(2023, 1, 2, 16, 25), 8)
(2, 'Motorista', 'Logística', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 30), datetime.datetime(2023, 1, 2, 19, 30), 8)
(3, 'Separador de Estoque', 'Estoque', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 9), datetime.datetime(2023, 1, 2, 16, 34), 8)


💡 **Os 3 parâmetros de `iter_rows(...)`:**
- `min_row=2` — comece a partir da linha 2 (pulando a linha 1, que é o cabeçalho com os títulos das colunas, não um dado de verdade).
- `max_row=4` — pare na linha 4 (senão o Python tentaria imprimir os 100 registros de uma vez).
- `values_only=True` — devolva só os valores de cada célula (ex.: `'Atendente'`), em vez de um "objeto célula" mais complexo que o Excel usa internamente. Sem esse parâmetro, você teria que escrever `celula.value` pra cada campo — `values_only=True` já entrega o valor pronto.

Não se preocupe com o formato `datetime.datetime(2023, 1, 2, 0, 0)` que aparece na saída — você vai entender exatamente o que é isso na próxima seção.

### ✏️ Atividade 1 — Contar por setor (CSV)

Usando `cadastro_funcionario.csv`, filtre e conte quantos funcionários existem no setor `"Estoque"`.

In [12]:
# (espaço para o código — construído ao vivo em aula)

import csv

with open("cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    funcionarios = list(csv.DictReader(arquivo))

funcionarios_estoque = [f for f in funcionarios if f["cargo"] == "Atendente"]
print(f"Existem {len(funcionarios_estoque)} funcionários no cargo de Atendente.")

Existem 7 funcionários no cargo de Atendente.


### ✏️ Atividade 2 — Escrever um resumo (JSON)

A partir do CSV, monte um dicionário contando quantos funcionários existem em cada setor e salve esse resumo em um novo arquivo `relatorio_setores.json`.

In [13]:
# (espaço para o código — construído ao vivo em aula)

contagem_setores = {}

for f in funcionarios:
    setor = f["setor"]
    contagem_setores[setor] = contagem_setores.get(setor, 0) + 1
    
resumo = {"total_funcionarios": len(funcionarios), "por setor":
    contagem_setores}

with open("relatorio_setores.json", "w", encoding="utf-8") as arquivo:
    json.dump(resumo, arquivo, ensure_ascii=False, indent=2)
    
print(resumo)

{'total_funcionarios': 30, 'por setor': {'Comercial': 8, 'Logística': 10, 'Estoque': 9, 'RH': 3}}


### ✏️ Atividade 3 — Explorar o Excel

Abra `recursos_humanos.xlsx`, confirme que a aba `banco_horas` tem 100 registros, e imprima os 5 primeiros.

In [24]:
# (espaço para o código — construído ao vivo em aula)

#importar XLSX
from openpyxl import load_workbook

planilha = load_workbook("recursos_humanos.xlsx")
aba_ponto = planilha["banco_horas"]

print(f"Total de registros é: {aba_ponto.max_row - 1}") # não inclui o cabeçalho

for linha in aba_ponto.iter_rows(min_row=2, max_row=6, values_only=True):
    print(linha)

Total de registros é: 100
(1, 'Atendente', 'Comercial', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 30), datetime.datetime(2023, 1, 2, 16, 25), 8)
(2, 'Motorista', 'Logística', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 30), datetime.datetime(2023, 1, 2, 19, 30), 8)
(3, 'Separador de Estoque', 'Estoque', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 9), datetime.datetime(2023, 1, 2, 16, 34), 8)
(4, 'Atendente', 'Comercial', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 9, 0), datetime.datetime(2023, 1, 2, 17, 15), 8)
(5, 'Motorista', 'Logística', datetime.datetime(2023, 1, 2, 0, 0), datetime.datetime(2023, 1, 2, 8, 5), datetime.datetime(2023, 1, 2, 18, 4), 8)


> ⚠️ **Atenção antes da próxima atividade.** Repare que, no resultado do Exemplo 1, `'id_funcionario': '1'` aparece **entre aspas** — isso significa que veio como texto, não como número. Isso acontece porque **todo valor lido de um CSV chega como texto**, mesmo que pareça um número. Já no JSON (Exemplo 2), `'id_funcionario': 1` aparece **sem aspas** — esse já é um número de verdade. Isso importa porque, em Python, `"9" == 9` é `False` — texto e número nunca são considerados iguais, mesmo representando "a mesma coisa" pros nossos olhos. Se for comparar o `id_funcionario` do CSV com o do JSON, primeiro transforme o do CSV em número com `int(...)`:
```python
id_do_csv = funcionarios[0]["id_funcionario"]   # '1' (texto)
id_convertido = int(id_do_csv)                    # 1 (número)
```

### ✏️ Atividade 4 — Cruzar duas fontes

Escolha um `id_funcionario`, busque o nome dele no CSV e o e-mail dele no JSON, e imprima os dois juntos (ex.: `"Igor Pereira: igor.pereira@grupoalfa.com.br"`). Lembre-se de converter o id do CSV com `int(...)` antes de comparar.

In [ ]:
# (espaço para o código — construído ao vivo em aula)
import csv
import json

# Ler arquivo CSV
with open("cadastro_funcionario.csv", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    funcionarios = list(leitor)
    
# Ler arquivo JSON 
with open("emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

# Atividade
id_escolhido = 9

nome = None
for f in funcionarios:
    if int(f["id_funcionario"]) == id_escolhido:
        nome = f["funcionario"]
        break
    
email = None
for e in emails:
    if e["id_funcionario"] == id_escolhido:
        email = e["email"]
        break
    
print(f"{nome}: {email}")

Igor Pereira: igor.pereira@grupoalfa.com.br


---
✅ **Checagem rápida — antes de avançar:**
Você consegue abrir o `cadastro_funcionario.csv` e contar quantos registros ele tem, sem olhar o exemplo?

---

---
## 2. Datas e Expressões Regulares

> 🔗 **Por que isso agora?** A aba `banco_horas` do Excel tem colunas de data e hora (`datahora_entrada`, `datahora_saida`). Pra calcular quanto cada funcionário trabalhou — ou se chegou atrasado — você precisa saber operar com essas datas. E o arquivo de e-mails tem 5 registros com formato quebrado, que só dá pra encontrar com um padrão de texto: é aí que entra o regex.

### Datas com `datetime`

### 🔹 Exemplo 1 — Calculando horas trabalhadas

💡 **Antes do código: como ler `datetime(2023, 1, 2, 8, 30)`?** Os 5 números são sempre nessa ordem: **ano, mês, dia, hora, minuto**. Aqui: 2023 (ano), 1 (janeiro), 2 (dia 2), 8 (8 horas), 30 (30 minutos) — ou seja, 2 de janeiro de 2023, às 8h30.

In [30]:
from datetime import datetime

# registro real de Ana Souza (id 1) em 02/01/2023
# ano, mês, dia, hora, minuto
entrada = datetime(2023, 1, 2, 8, 30)
saida = datetime(2023, 1, 2, 16, 25)

horas_trabalhadas = (saida - entrada).seconds / 3600
print(f"Horas trabalhadas: {horas_trabalhadas:.2f}h")

Horas trabalhadas: 7.92h


💡 **O que aconteceu em cada linha:**
- Quando você subtrai duas datas/horas em Python (`saida - entrada`), o resultado **não é um número** — é um "pacote" especial chamado `timedelta` ("diferença de tempo"), que guarda quantos dias e quantos segundos existem entre as duas datas.
- `.seconds` pega só a parte em segundos desse pacote. Como `entrada` e `saida` são no mesmo dia, isso já nos dá o total certo.
- 1 hora tem 3600 segundos (60 minutos × 60 segundos) — por isso dividimos por `3600`: é a conversão de segundos para horas.
- `:.2f` dentro do `f"..."` significa "mostre só 2 casas depois da vírgula". Sem isso, apareceria algo como `7.916666666666667` — difícil de ler.

### 🔹 Exemplo 2 — Calculando o atraso em minutos

In [34]:
from datetime import datetime

entrada_prevista_min = 8 * 60  # 08:00 que são 480 minutos
entrada_real = datetime(2023, 1, 2, 8, 30) # ano, mês, dia, hora, minuto
entrada_real_min = entrada_real.hour * 60 + entrada_real.minute

atraso_minutos = entrada_real_min - entrada_prevista_min

print(entrada_prevista_min, "minutos")
print(entrada_real.hour, "horas")
print(entrada_real.minute, "minutos")
print(entrada_real_min, "minutos")

print(f"\nAtraso: {atraso_minutos} minutos")

480 minutos
8 horas
30 minutos
510 minutos

Atraso: 30 minutos


> ⚠️ **Veja como é um erro real do Python.** Tente criar `datetime(2023, 13, 1)` (mês 13 não existe) numa célula nova — você vai ver `ValueError: month must be in 1..12`. Diferente do `FileNotFoundError` (um arquivo que não existe), esse é um erro de **valor inválido**: a data em si não existe no calendário.

### ✏️ Atividade 5 — Outro registro

Usando o Excel, pegue o registro de outro `id_funcionario` do dia 02/01/2023 e calcule as horas trabalhadas dele.

In [ ]:
# (espaço para o código — construído ao vivo em aula)

from openpyxl import load_workbook
from datetime import datetime

planilha = load_workbook("recursos_humanos.xlsx")

# ano, mês, dia, hora, minuto
entrada = datetime(2023, 1, 2, 8, 9)
saida = datetime(2023, 1, 2, 16, 34)

horas_trabalhadas = (saida - entrada).seconds / 3600
print(f"Horas trabalhadas: {horas_trabalhadas:.2f}h")

### ✏️ Atividade 6 — Atraso de outro funcionário

Calcule o atraso (em minutos) do mesmo registro da Atividade 5, comparando com a entrada prevista das 08:00. Se o resultado for negativo, o que isso significa?

In [37]:
# (espaço para o código — construído ao vivo em aula)

entrada_prevista_min = 8 * 60  # 08:00 que são 480 minutos
entrada_real = datetime(2023, 1, 2, 7, 45) # ano, mês, dia, hora, minuto
entrada_real_min = entrada_real.hour * 60 + entrada_real.minute

atraso_minutos = entrada_real_min - entrada_prevista_min

print(entrada_prevista_min, "minutos")
print(entrada_real.hour, "horas")
print(entrada_real.minute, "minutos")
print(entrada_real_min, "minutos")

print(f"\nAtraso: {atraso_minutos} minutos, ou seja, chegou adiantado.")

480 minutos
7 horas
45 minutos
465 minutos

Atraso: -15 minutos, ou seja, chegou adiantado.


### Expressões regulares com `re`

Uma expressão regular (regex) é um padrão de texto usado para **validar** ou **buscar** formatos específicos. Você já sabe que 5 dos 30 e-mails de `emails_corporativos.json` vieram quebrados do sistema de TI — é hora de encontrá-los.

### 🔹 Exemplo 1 — Validando um e-mail

💡 **Antes do código: o que é o "r" antes das aspas?** Repare que o padrão vai começar com `r"..."` — esse `r` colado nas aspas cria o que chamamos de *raw string* ("texto bruto"). Padrões de regex usam bastante a barra invertida (`\`), e sem o `r` o Python tentaria entender essa barra de outro jeito, o que quebraria o padrão. **Regra prática: sempre que for escrever um padrão de regex, comece com `r"..."`.**

In [38]:
import re    # importando regex

email = "ana.souza@grupoalfa.com.br"
padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"


if re.match(padrao, email):
    print("E-mail parece válido!")
else:
    print("E-mail inválido.")

E-mail parece válido!


### 🔹 Exemplo 2 — Separando todos os e-mails válidos dos inválidos

In [53]:
import json, re

with open("emails_corporativos.json", encoding="utf-8") as arquivo:
    emails = json.load(arquivo)

padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"

validos = []
invalidos = []
for registro in emails:
    if re.match(padrao, registro["email"]):
        validos.append(registro)        
    else:
        invalidos.append(registro)
        

print(f"E-mails válidos: {len(validos)}")
print(f"E-mails inválidos: {len(invalidos)}")

print("\nE-mails inválidos:")
for i in invalidos:
    print(i["email"])
    
print("\nE-mails válidos:")
for v in validos:
    print(v["email"])
    


E-mails válidos: 25
E-mails inválidos: 5

E-mails inválidos:
carlos.mendesgrupoalfa.com.br
helena martins@grupoalfa.com.br
paulo.gomes@@grupoalfa.com.br
vinicius.barbosa@
andre.batista@grupoalfacombr

E-mails válidos:
ana.souza@grupoalfa.com.br
bruno.lima@grupoalfa.com.br
daniela.rocha@grupoalfa.com.br
eduardo.santos@grupoalfa.com.br
fernanda.costa@grupoalfa.com.br
gabriel.oliveira@grupoalfa.com.br
igor.pereira@grupoalfa.com.br
juliana.neves@grupoalfa.com.br
lucas.alves@grupoalfa.com.br
mariana.castro@grupoalfa.com.br
nathan.ribeiro@grupoalfa.com.br
olivia.ferreira@grupoalfa.com.br
queila.dias@grupoalfa.com.br
rafael.teixeira@grupoalfa.com.br
sabrina.moura@grupoalfa.com.br
thiago.cardoso@grupoalfa.com.br
ursula.pinto@grupoalfa.com.br
wanderson.cruz@grupoalfa.com.br
ximena.lopes@grupoalfa.com.br
yago.nascimento@grupoalfa.com.br
zilda.campos@grupoalfa.com.br
beatriz.cunha@grupoalfa.com.br
caio.freitas@grupoalfa.com.br
debora.azevedo@grupoalfa.com.br
emerson.tavares@grupoalfa.com.br


> ⚠️ **Cuidado com padrões "soltos".** Um padrão mais simples como `r".+@.+\..+"` (sem `^`/`$` e sem excluir espaços) parece funcionar, mas na prática aceita coisas erradas — inclusive o e-mail quebrado `"helena martins@grupoalfa.com.br"` (tem espaço) passaria como válido. Teste sempre com um caso que **deveria falhar**, não só com um caso válido.

### ✏️ Atividade 7 — Quem precisa corrigir o e-mail

Imprima o nome de cada funcionário da lista `invalidos`, pra avisar o time de TI.

In [56]:
# (espaço para o código — construído ao vivo em aula)

for i in invalidos:
    print(f"{i["funcionario"]}: {i["email"]}")

Carlos Mendes: carlos.mendesgrupoalfa.com.br
Helena Martins: helena martins@grupoalfa.com.br
Paulo Gomes: paulo.gomes@@grupoalfa.com.br
Vinicius Barbosa: vinicius.barbosa@
André Batista: andre.batista@grupoalfacombr


### ✏️ Atividade 8 — Seu próprio e-mail

Monte uma variável com um e-mail no padrão `nome.sobrenome@grupoalfa.com.br` usando o seu nome, e teste se ele passa no padrão.

In [64]:
# (espaço para o código — construído ao vivo em aula)
import re

meu_email = 'leislaa@gmail.com.br'

padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
# texto + @ + texto + . + texto

if re.match(padrao, meu_email):
    print("E-mail parece válido!")
else:
    print("E-mail inválido.")

E-mail parece válido!


---
✅ **Checagem rápida — antes de avançar:**
Você consegue calcular quantos minutos de atraso um registro do banco de horas teve, e dizer quantos e-mails do JSON são inválidos?

---

---
## 3. Funções: parâmetros, valores padrão e `return`

> 🔗 **Por que isso agora?** Você já sabe calcular horas trabalhadas e atraso de um registro. Mas se precisar repetir esse cálculo pra outro funcionário — ou pros 100 registros do banco de horas — teria que copiar e colar o código de novo, só trocando os valores. É exatamente esse problema que uma função resolve.

Olhe este código que calcula o atraso de 3 funcionários diferentes, copiado e colado 3 vezes:
```python
atraso_ana = (8*60 + 30) - (8*60)      # 30 min
atraso_bruno = (8*60 + 30) - (8*60)    # 30 min
atraso_carlos = (8*60 + 9) - (8*60)    # 9 min
```
A mesma conta (`entrada_real − entrada_prevista`, em minutos) está escrita 3 vezes — se a tolerância mudar um dia, é fácil esquecer de corrigir uma das linhas. Uma função resolve isso: você escreve a regra **uma única vez**.

### 📖 Antes da função completa: a menor função possível em Python

```python
def dizer_ola():
    return "Olá!"

print(dizer_ola())    # Olá!
```

Repare na anatomia — toda função em Python segue essa mesma receita:
- `def` é a palavra que avisa o Python "a partir daqui eu estou **de**finindo uma função nova" (abreviação de "definir").
- `dizer_ola` é o nome que você escolheu para a função — mesma regra de nomes de variáveis que você já usa.
- `()` é onde ficam os **parâmetros** — "caixinhas vazias" que a função pode pedir pra receber um valor toda vez que for chamada. Aqui está vazio porque essa função não precisa de nenhuma informação de fora.
- A linha indentada (recuada) é o **corpo** da função — o que ela faz de fato.
- `return` é a palavra que diz "é isso que a função devolve pra quem chamou ela".

Agora a mesma receita, com uma "caixinha" (parâmetro) pra preencher:

```python
def dizer_ola_para(nome):
    return f"Olá, {nome}!"

print(dizer_ola_para("Ana"))    # Olá, Ana!
```

É essa mesma receita que a função `calcular_atraso` abaixo usa — só que com 4 caixinhas (parâmetros) em vez de 1, e duas delas já vêm com um valor "pré-preenchido" (o valor padrão) caso você não informe nada.

In [ ]:
def dizer_ola():
    return "Olá!"

print(dizer_ola())

def dizer_ola_para(nome):
    return f"Olá, {nome}!"

print(dizer_ola_para("Ana"))

In [ ]:
def calcular_atraso(hora_entrada, minuto_entrada, hora_prevista=8, minuto_previsto=0):
    minutos_reais = hora_entrada * 60 + minuto_entrada
    minutos_previstos = hora_prevista * 60 + minuto_previsto
    return minutos_reais - minutos_previstos

print(calcular_atraso(8, 30))        # entrada 08:30, previsto padrão 08:00 → 30
print(calcular_atraso(7, 45))        # chegou antes → -15

💡 **"Sem `return`, a função é surda — não responde nada."** Uma função sem `return` pode até imprimir algo na tela, mas não devolve nenhum valor que você possa guardar em uma variável ou usar em outro cálculo.

### 🔹 Exemplo 1 — Função com retorno: horas trabalhadas

In [ ]:
from datetime import datetime

def calcular_horas_trabalhadas(entrada, saida):
    return (saida - entrada).seconds / 3600

entrada = datetime(2023, 1, 2, 8, 30)
saida = datetime(2023, 1, 2, 16, 25)
print(f"{calcular_horas_trabalhadas(entrada, saida):.2f}h")

### 🔹 Exemplo 2 — Parâmetro com valor padrão: classificar pontualidade

In [ ]:
def classificar_pontualidade(atraso_minutos, tolerancia=10):
    if atraso_minutos <= tolerancia:
        return "Pontual"
    return "Atrasado"

print(classificar_pontualidade(5))                   # dentro da tolerância padrão: Pontual
print(classificar_pontualidade(30))                  # acima da tolerância padrão: Atrasado
print(classificar_pontualidade(12, tolerancia=15))   # tolerância customizada: Pontual

> ⚠️ **Veja como é um erro real do Python.** Chame `calcular_horas_trabalhadas()` sem passar os dois parâmetros numa célula nova — você vai ver `TypeError: calcular_horas_trabalhadas() missing 2 required positional arguments: 'entrada' and 'saida'`. Diferente de `tolerancia` (que tem valor padrão e pode ser omitido), `entrada` e `saida` não têm padrão — por isso são **obrigatórios**.

### ✏️ Atividade 9 — `calcular_horas_trabalhadas`

Crie a função e teste com 2 registros diferentes do banco de horas.

In [ ]:
# (espaço para o código — construído ao vivo em aula)

### ✏️ Atividade 10 — `calcular_atraso` com parâmetro padrão

Crie a função `calcular_atraso(hora_entrada, minuto_entrada, hora_prevista=8, minuto_previsto=0)` e teste uma vez com a entrada prevista padrão (08:00) e outra vez com um horário customizado (ex.: turno que começa às 07:00).

In [ ]:
# (espaço para o código — construído ao vivo em aula)

### ✏️ Atividade 11 — `validar_email`

Transforme a checagem de regex da Seção 2 em uma função `validar_email(email)` que retorna `True` ou `False`.

In [ ]:
# (espaço para o código — construído ao vivo em aula)

### ✏️ Atividade 12 — `classificar_pontualidade`

Crie a função (como no Exemplo 2) e teste com 3 valores diferentes de atraso.

In [ ]:
# (espaço para o código — construído ao vivo em aula)

> 🚀 **Avançado (opcional) — parâmetros "ilimitados" com `*args`:** toda função até aqui tinha um número fixo de parâmetros. Com `*args`, uma função aceita **qualquer quantidade** de argumentos, agrupados numa tupla:
> ```python
> def media(*args):
>     return sum(args) / len(args)
>
> print(media(7, 8, 9))        # funciona com 3 valores
> print(media(7, 8, 9, 10))    # e também com 4, sem mudar a função
> ```
> Isso é a base do desafio opcional ao final desta seção.

---
✅ **Checagem rápida — antes de avançar:**
Você consegue escrever uma função com um parâmetro obrigatório e um parâmetro com valor padrão, e explicar a diferença entre os dois?

---

---
## 4. Lambda e Módulos

> 🔗 **Por que isso agora?** Uma `lambda` não é um conceito novo — é só um jeito mais curto de escrever uma função pequena como as que você acabou de criar. E um módulo é só um arquivo com as suas funções guardadas, prontas pra reusar sem copiar e colar.

### 🔹 Exemplo 1 — Função lambda

In [ ]:
calcular_atraso_lambda = lambda entrada_min, previsto_min=8*60: entrada_min - previsto_min

print(calcular_atraso_lambda(8*60 + 30))    # 30

> ⚠️ **Onde a lambda esbarra.** Tente escrever `lambda x: return x * 2` numa célula nova — o Python recusa com `SyntaxError: invalid syntax`. Lambda só aceita uma **expressão** (algo que produz um valor), nunca um comando como `return`, `if` sozinho ou um `for`. É por isso que lambda serve pra cálculos curtos, não pra lógica complexa.

### 🔹 Exemplo 2 — Onde o lambda realmente brilha: dentro de `min()`/`sorted()`

Guardar uma lambda numa variável (Exemplo 1) é raro na prática. O uso real é passar a lambda **direto** como argumento de outra função, pra dizer "compare por *este* critério":

In [ ]:
from datetime import datetime

# registros reais do dia 02/01/2023 (banco_horas)
registros_do_dia = [
    {"funcionario": "Ana Souza",      "entrada": datetime(2023, 1, 2, 8, 30)},
    {"funcionario": "Bruno Lima",     "entrada": datetime(2023, 1, 2, 8, 30)},
    {"funcionario": "Helena Martins", "entrada": datetime(2023, 1, 2, 7, 25)},
]

primeiro_a_chegar = min(registros_do_dia, key=lambda r: r["entrada"])
print(f"Primeiro a chegar: {primeiro_a_chegar['funcionario']}")

### Criando seu próprio módulo

Tanto no Colab quanto no VS Code (rodando este notebook com a extensão Jupyter), criamos o "módulo" como um arquivo de texto usando o comando mágico `%%writefile` — ele funciona nos dois ambientes porque ambos rodam sobre um kernel Jupyter, e cria o arquivo `utilidades_rh.py` na mesma pasta do notebook.

> 📁 **Onde o arquivo é criado?** No Colab, dentro da pasta temporária `/content` (some quando a sessão termina — por isso reenvie os 3 arquivos originais se voltar depois). No VS Code local, na mesma pasta do notebook, como um arquivo `.py` de verdade — permanente, e que você pode (e deve) versionar no Git.

In [ ]:
%%writefile utilidades_rh.py
def calcular_horas_trabalhadas(entrada, saida):
    return (saida - entrada).seconds / 3600

def calcular_atraso(hora_entrada, minuto_entrada, hora_prevista=8, minuto_previsto=0):
    minutos_reais = hora_entrada * 60 + minuto_entrada
    minutos_previstos = hora_prevista * 60 + minuto_previsto
    return minutos_reais - minutos_previstos

def classificar_pontualidade(atraso_minutos, tolerancia=10):
    if atraso_minutos <= tolerancia:
        return "Pontual"
    return "Atrasado" 

In [ ]:
import utilidades_rh

atraso = utilidades_rh.calcular_atraso(8, 30)
print(utilidades_rh.classificar_pontualidade(atraso))

### ✏️ Atividade 13 — Lambda

Reescreva `calcular_horas_trabalhadas` (Seção 3) como uma `lambda`.

In [ ]:
# (espaço para o código — construído ao vivo em aula)

### ✏️ Atividade 14 — Ordenando por critério

Monte uma lista com 3-4 registros de entrada (pode reaproveitar dados reais do banco de horas) e use `sorted()` com `key=lambda` pra ordenar do que chegou mais cedo ao mais tarde.

In [ ]:
# (espaço para o código — construído ao vivo em aula)

### ✏️ Atividade 15 — Adicionar uma função ao módulo

Adicione a função `validar_email` (Atividade 11) ao arquivo `utilidades_rh.py` (reescreva o arquivo com `%%writefile`, incluindo as funções anteriores + a nova).

In [ ]:
%%writefile utilidades_rh.py
import re

def calcular_horas_trabalhadas(entrada, saida):
    return (saida - entrada).seconds / 3600

def calcular_atraso(hora_entrada, minuto_entrada, hora_prevista=8, minuto_previsto=0):
    minutos_reais = hora_entrada * 60 + minuto_entrada
    minutos_previstos = hora_prevista * 60 + minuto_previsto
    return minutos_reais - minutos_previstos

def classificar_pontualidade(atraso_minutos, tolerancia=10):
    if atraso_minutos <= tolerancia:
        return "Pontual"
    return "Atrasado"

def validar_email(email):
    padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
    return bool(re.match(padrao, email))

### ✏️ Atividade 16 — Usando o módulo reimportado

Reimporte o módulo (com `importlib.reload`, já que ele foi reescrito) e use `validar_email` para processar os e-mails de `emails_corporativos.json`.

In [ ]:
import importlib
import utilidades_rh
importlib.reload(utilidades_rh)  # recarrega o módulo já que ele foi reescrito

# (espaço para o código — construído ao vivo em aula)

---
✅ **Checagem rápida — antes de avançar:**
Você consegue explicar em uma frase por que uma lambda não pode ter `return` dentro dela?

---

---
### 🏁 Fechamento — Semana 04

**Nesta semana você aprendeu:**
- Leitura e escrita de CSV, JSON e Excel — a base para trabalhar com dados reais de qualquer sistema
- Funções bem definidas: parâmetros, `return`, lambda e módulos
- Datas e expressões regulares para lidar com dados de ponto e cadastro

**Próxima semana:** Semana 05 — Pandas e NumPy: as ferramentas centrais do analista de dados.

---
## 5. Treino em Squads — Sexta-feira (Encontro 3)

Esta seção é usada **em sala (ou em salas remotas/breakout)** na sexta-feira. Cada squad recebe **um setor do Grupo Alfa** (Comercial, Logística, Estoque ou RH) e precisa consolidar as 3 fontes de dados pra responder uma pergunta real de People Analytics:

> **Desafio:** usando os 3 arquivos (`cadastro_funcionario.csv`, `emails_corporativos.json`, `recursos_humanos.xlsx`), filtre só os funcionários do setor do seu squad, descubra **quem teve mais atraso acumulado** na semana de dados disponível, e verifique se o e-mail dessa pessoa está correto. Organizem o código do squad em pelo menos uma função reutilizável.

In [ ]:
# SQUAD — troque "Comercial" pelo setor que seu squad recebeu: Comercial, Logística, Estoque ou RH
SETOR_DO_SQUAD = "Comercial"

# escrevam o código do squad aqui: filtrar por setor, somar atraso por funcionário,
# achar quem teve mais atraso, e validar o e-mail dessa pessoa


### 🗣️ Debate coletivo (após as apresentações)

Depois que todos os squads apresentarem, discuta com a turma:

- Os squads chegaram ao mesmo resultado pro setor deles?
- Alguém encontrou um funcionário com e-mail inválido?
- As funções ficaram parecidas entre os squads, ou cada um organizou diferente?

### Desafio (opcional)

Reescreva a função `calcular_horas_trabalhadas` usando `*args` pra calcular a média de horas trabalhadas em vários registros de uma vez, sem passar uma lista.

---
### Assinatura

Curso: **Análise de Dados com Python — SENAI (Turma T5)**
Semana 04 — Manipulação de Arquivos e Modularização

*Prof. Especialista Cláudio F. Neves*